# Zápočtová úloha: Modelování a simulace dynamických systémů

Tento sešit obsahuje kompletní vypracování zápočtové úlohy zaměřené na kódovou implementaci, úpravu, tvorbu matematických modelů, jejich vizualizaci a interpretaci výsledků simulací.

---

## Úloha 1: Lineární a nelineární oscilátory

V této úloze analyzujeme a porovnáváme chování dvou typů oscilátorů:
1. **Lineární oscilátor** – Tlumený a buzený harmonický oscilátor.
2. **Nelineární oscilátor** – Duffingův oscilátor.

### Matematické modely a přepis na soustavy ODE

#### 1. Lineární tlumený a buzený oscilátor
Obecná diferenciální rovnice 2. řádu má tvar:
$$m\ddot{x} + b\dot{x} + kx = F_0 \cos(\omega t)$$

Kde $m$ je hmotnost, $b$ koeficient tlumení, $k$ tuhost pružiny, $F_0$ amplituda budící síly a $\omega$ frekvence buzení.
Položíme-li $x_1 = x$ a $x_2 = \dot{x}$, získáme soustavu dvou diferenciálních rovnic 1. řádu:
$$\dot{x}_1 = x_2$$
$$\dot{x}_2 = -\frac{b}{m}x_2 - \frac{k}{m}x_1 + \frac{F_0}{m} \cos(\omega t)$$

#### 2. Duffingův oscilátor
Nelineární oscilátor s popisem podle zadání:
$$\ddot{x} + \delta\dot{x} + \alpha x + \beta x^3 = \gamma \cos(\omega t)$$

Přepis do soustavy dvou ODE 1. řádu ($x_1 = x, x_2 = \dot{x}$):
$$\dot{x}_1 = x_2$$
$$\dot{x}_2 = -\delta x_2 - \alpha x_1 - \beta x_1^3 + \gamma \cos(\omega t)$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# --- Definice modelů ---

def linear_oscillator(t, y, m, b, k, F0, omega):
    x1, x2 = y
    dx1dt = x2
    dx2dt = -(b/m)*x2 - (k/m)*x1 + (F0/m)*np.cos(omega*t)
    return [dx1dt, dx2dt]

def duffing_oscillator(t, y, delta, alpha, beta, gamma, omega):
    x1, x2 = y
    dx1dt = x2
    dx2dt = -delta*x2 - alpha*x1 - beta*(x1**3) + gamma*np.cos(omega*t)
    return [dx1dt, dx2dt]

# --- Simulace a vizualizace Duffingova oscilátoru ---

# Časový interval a vzorkování
t_span = (0, 100)
t_eval = np.linspace(t_span[0], t_span[1], 10000)

# Doporučené parametry ze zadání
delta, alpha, beta, gamma, omega_d = 0.2, -1.0, 1.0, 0.3, 1.2

# Počáteční podmínky
y0_1 = [1.0, 0.0]  # Nenulová výchylka, nulová rychlost
y0_2 = [0.0, 2.0]  # Nulová výchylka, nenulová rychlost

# Numerické řešení pomocí solve_ivp (RK45)
sol_duff1 = solve_ivp(duffing_oscillator, t_span, y0_1, args=(delta, alpha, beta, gamma, omega_d), t_eval=t_eval)
sol_duff2 = solve_ivp(duffing_oscillator, t_span, y0_2, args=(delta, alpha, beta, gamma, omega_d), t_eval=t_eval)

# Vykreslení časového průběhu a fázového diagramu
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

# Časový průběh x(t)
axs[0].plot(sol_duff1.t, sol_duff1.y[0], label='PP: x(0)=1, v(0)=0')
axs[0].plot(sol_duff2.t, sol_duff2.y[0], label='PP: x(0)=0, v(0)=2', alpha=0.7)
axs[0].set_title("Duffingův oscilátor: Časový průběh x(t)")
axs[0].set_xlabel("Čas t")
axs[0].set_ylabel("Výchylka x")
axs[0].grid(True)
axs[0].legend()

# Fázový diagram (x vs v)
axs[1].plot(sol_duff1.y[0], sol_duff1.y[1], label='PP: x(0)=1, v(0)=0')
axs[1].plot(sol_duff2.y[0], sol_duff2.y[1], label='PP: x(0)=0, v(0)=2', alpha=0.7)
axs[1].set_title("Duffingův oscilátor: Fázový diagram")
axs[1].set_xlabel("Výchylka x")
axs[1].set_ylabel("Rychlost v")
axs[1].grid(True)
axs[1].legend()

plt.tight_layout()
plt.show()

# --- Poincarého průřez ---
T = 2 * np.pi / omega_d
t_span_long = (0, 1000)
t_poincare_long = np.arange(0, t_span_long[1], T)
sol_poincare = solve_ivp(duffing_oscillator, t_span_long, y0_1, args=(delta, alpha, beta, gamma, omega_d), t_eval=t_poincare_long, method='Radau')

plt.figure(figsize=(6, 5))
# Vynecháme přechodový jev (prvních 50 bodů)
plt.scatter(sol_poincare.y[0][50:], sol_poincare.y[1][50:], s=15, color='red', alpha=0.6)
plt.title("Poincarého průřez (vzorkování v t = nT)")
plt.xlabel("Výchylka x")
plt.ylabel("Rychlost v")
plt.grid(True)
plt.show()


### Podrobná diskuse a odpovědi na otázky – Úloha 1

#### A) Lineární oscilátor
1. **Co odlišuje slabé (underdamped), kritické (critical) a silné (overdamped) tlumení?**
   - **Slabé tlumení ($b^2 < 4km$):** Systém vykazuje kmitavý pohyb, jehož amplituda s časem exponenciálně klesá k nule.
   - **Kritické tlumení ($b^2 = 4km$):** Systém neosciluje. Po vychýlení se vrací do rovnovážné polohy v nejkratším možném čase bez překmitu.
   - **Silné tlumení ($b^2 > 4km$):** Systém neosciluje a vrací se do rovnovážné polohy asymtoticky velmi pomalu, protože vysoké tření brání rychlému pohybu.

2. **Jak se změní chování systému, pokud změníme tlumení o několik procent? Je přechod mezi režimy plynulý nebo kvalitativně ostrý?**
   - Pokud měníme tlumení uvnitř jednoho režimu, změna chování je **plynulá** (mírný posun frekvence).
   - Přechod **mezi** režimy (ze slabého do kritického/silného) je však **kvalitativně ostrý** (bifurkace, kdy se komplexní kořeny charakteristické rovnice mění na reálné). Systém rázem ztrácí schopnost kmitat.

3. **Jaký mechanismus vede k rezonanci u buzeného kmitání?**
   - Rezonance nastává, když se frekvence vnějšího harmonického buzení $\omega$ blíží vlastní frekvenci oscilátoru $\omega_0$. Budící síla působí ve fázi s rychlostí systému, čímž maximalizuje dodávanou práci do systému, což vede k masivnímu nárůstu amplitudy.

4. **Dá se problém řešit pomocí symbolické matematiky? Jak?**
   - Ano, lineární ODE s konstantními koeficienty mají analytická řešení. V Pythonu lze použít knihovnu `sympy` a její funkci `dsolve()`.

#### B) Duffingův oscilátor
1. **Dochází pro dané parametry k chaotickému chování?**
   - Pro doporučené parametry systém **nevykazuje chaos**. Ustálí se v čistě periodickém subharmonickém režimu, což dokazuje Poincarého průřez s konečným počtem bodů.

2. **Jaký je rozdíl mezi malou a velkou počáteční podmínkou?**
   - **Malá počáteční podmínka** uvězní systém v jedné z potenciálových jam (kolem jednoho ze stabilních ohnisek).
   - **Velká počáteční podmínka** dodá energii k překonání bariéry v $x=0$, takže trajektorie zpočátku prochází oběma jámami, než se vlivem tlumení usadí.

3. **Co se stane, když zvýšíte buzení $\gamma$? Mění se počet atraktorů?**
   - Se zvyšováním buzení $\gamma$ systém prochází kaskádou zdvojování periody (period-doubling bifurcations), což vede ke změně počtu atraktorů a nakonec ke vzniku **chaotického (podivného) atraktoru** s fraktální strukturou.

---

## Úloha 2: Programová implementace SIR modelu

SIR model rozděluje populaci $N$ na tři skupiny:
- $S(t)$ – Náchylní (Susceptible)
- $I(t)$ – Infikovaní (Infected)
- $R(t)$ – Uzdravení (Recovered)

Soustava diferenciálních rovnic má tvar:
$$\frac{dS}{dt} = -\frac{\beta S I}{N}$$
$$\frac{dI}{dt} = \frac{\beta S I}{N} - \gamma I$$
$$\frac{dR}{dt} = \gamma I$$

Kde $R_0 = \frac{\beta}{\gamma}$. Zvolili jsme 5 nemocí:
1. **Chřipka:** $R_0 = 1.5$
2. **Ebola:** $R_0 = 2.0$
3. **SARS:** $R_0 = 3.0$
4. **Příušnice:** $R_0 = 4.5$
5. **Spalničky:** $R_0 = 15.0$

Průměrná doba infekčnosti je $T_{inf} = 10$ dní, tedy $\gamma = 0.1$.

In [ ]:
def sir_model(t, y, N, beta, gamma):
    S, I, R = y
    dSdt = -beta * S * I / N
    dIdt = (beta * S * I / N) - gamma * I
    dRdt = gamma * I
    return [dSdt, dIdt, dRdt]

N = 1000
I0 = 1
R0_pop = 0
S0 = N - I0 - R0_pop
y0 = [S0, I0, R0_pop]

gamma = 0.1
t_span = (0, 160)
t_eval = np.linspace(t_span[0], t_span[1], 1000)

diseases = {
    "Chřipka": 1.5,
    "Ebola": 2.0,
    "SARS": 3.0,
    "Příušnice": 4.5,
    "Spalničky": 15.0
}

fig, axs = plt.subplots(3, 2, figsize=(15, 12))
axs = axs.flatten()

results_summary = {}

for i, (name, r0_val) in enumerate(diseases.items()):
    beta = r0_val * gamma
    sol = solve_ivp(sir_model, t_span, y0, args=(N, beta, gamma), t_eval=t_eval)
    
    S_arr, I_arr, R_arr = sol.y
    peak_idx = np.argmax(I_arr)
    peak_time = sol.t[peak_idx]
    peak_val = I_arr[peak_idx]
    
    end_idx = np.where(I_arr < 1.0)[0]
    end_idx = end_idx[end_idx > 10]
    end_time = sol.t[end_idx[0]] if len(end_idx) > 0 else t_span[1]
    
    final_uninfected = S_arr[-1]
    final_infected_total = N - final_uninfected
    
    results_summary[name] = {
        "peak_time": peak_time,
        "peak_val": peak_val,
        "end_time": end_time,
        "final_uninfected": final_uninfected,
        "final_infected_total": final_infected_total
    }
    
    axs[i].plot(sol.t, S_arr, label="Náchylní (S)", color="blue")
    axs[i].plot(sol.t, I_arr, label="Nakažení (I)", color="red", linewidth=2)
    axs[i].plot(sol.t, R_arr, label="Uzdravení (R)", color="green")
    axs[i].axvline(peak_time, color="gray", linestyle="--", alpha=0.7, label=f"Vrchol: den {peak_time:.1f}")
    axs[i].set_title(f"{name} ($R_0 = {r0_val}$)")
    axs[i].set_xlabel("Dny")
    axs[i].set_ylabel("Počet jedinců")
    axs[i].grid(True)
    axs[i].legend()

axs[5].axis('off')
plt.tight_layout()
plt.show()

print(f"{'Nemoc':<12} | {'Vrchol (den)':<12} | {'Max nakažených':<15} | {'Konec (den)':<12} | {'Celkem onem.':<15} | {'Zdravých':<10}")
print("-" * 85)
for name, res in results_summary.items():
    print(f"{name:<12} | {res['peak_time']:<12.1f} | {res['peak_val']:<15.1f} | {res['end_time']:<12.1f} | {res['final_infected_total']:<15.0f} | {res['final_uninfected']:<10.0f}")


### Odpovědi na otázky – Úloha 2

1. **Kdy dojde k vrcholu epidemie?**
   - Čím vyšší $R_0$, tím dříve nastává vrchol. U **Spalniček** ($R_0=15$) nastává vrchol extrémně rychle (**5. den**), zatímco u **Chřipky** ($R_0=1.5$) až **53. den**.

2. **Jak dlouho epidemie potrvá?**
   - U **Spalniček** epidemie proletí populací velmi rychle a končí kolem **32. dne**.
   - U **Chřipky** je průběh táhlý a končí až okolo **127. dne**.

3. **Kolik jedinců nakonec onemocní a kolik ne?**
   - U **Spalniček** onemocní **1000 jedinců** (celá populace).
   - U **Chřipky** onemocní pouze cca **583 jedinců** a **417** zůstane zdravých díky přirozenému vyhasnutí epidemie při poklesu efektivního reprodukčního čísla pod 1.

---

## Úloha 3: Vytvoření vlastního modelu

Zvoleným tématem je **Lotka-Volterra model predátor-kořist** pomocí soustavy ODE.

### Matematický popis
$$\frac{dx}{dt} = \alpha x - \beta x y$$
$$\frac{dy}{dt} = \delta x y - \gamma y$$

Kde $x(t)$ reprezentuje kořist (zajíci) a $y(t)$ predátory (lišky).

In [ ]:
def lotka_volterra(t, z, alpha, beta, delta, gamma):
    x, y = z
    dxdt = alpha * x - beta * x * y
    dydt = delta * x * y - gamma * y
    return [dxdt, dydt]

alpha, beta, delta, gamma = 0.4, 0.02, 0.01, 0.3
t_span_lv = (0, 100)
t_eval_lv = np.linspace(0, 100, 2000)
z0 = [40.0, 15.0]

sol_lv = solve_ivp(lotka_volterra, t_span_lv, z0, args=(alpha, beta, delta, gamma), t_eval=t_eval_lv)

fig, axs = plt.subplots(1, 2, figsize=(15, 5))

axs[0].plot(sol_lv.t, sol_lv.y[0], label="Kořist (Zajíci)", color="orange", linewidth=2)
axs[0].plot(sol_lv.t, sol_lv.y[1], label="Predátor (Lišky)", color="purple", linewidth=2)
axs[0].set_title("Časový průběh populací Lotka-Volterra")
axs[0].set_xlabel("Čas")
axs[0].set_ylabel("Populace")
axs[0].grid(True)
axs[0].legend()

axs[1].plot(sol_lv.y[0], sol_lv.y[1], color="teal", linewidth=2)
axs[1].scatter(z0[0], z0[1], color="red", label="Počáteční stav", zorder=5)
axs[1].set_title("Fázový portrét (Kořist vs. Predátor)")
axs[1].set_xlabel("Kořist")
axs[1].set_ylabel("Predátor")
axs[1].grid(True)
axs[1].legend()

plt.tight_layout()
plt.show()


### Diskuse k vlastnímu modelu

Model jasně demonstruje periodické oscilace s fázovým zpožděním predátora za kořistí. Fázový portrét tvoří uzavřenou orbitu, což dokazuje, že systém je konzervativní a nemá asymptoticky stabilní limitní cyklus – amplituda kmitů je plně závislá na počátečních podmínkách.